In [1]:
# === Ô GHI LOG (mẫu PL1 bản 2 – chạy được với ipykernel 7): mọi thứ in ra màn hình đồng thời được ghi vào file log ===
import logging, sys, datetime, warnings
from pathlib import Path
TEN_NOTEBOOK = "00_E0.1_cach_ghi_va_thoi_gian_Han"
NGUOI_CHAY   = "Han"

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())  # thư mục gốc repo
thu_muc_log = ROOT / "logs" / "eda_log"
thu_muc_log.mkdir(parents=True, exist_ok=True)
file_log = thu_muc_log / f"{datetime.datetime.now():%Y-%m-%d_%H%M}_{TEN_NOTEBOOK}.log"
tep_log = open(file_log, "a", encoding="utf-8", buffering=1)

if hasattr(sys.stdout, "echo"):                  # chạy trong Jupyter / VS Code: nhờ kernel chép thêm ra file, không thay sys.stdout
    sys.stdout.echo = tep_log
    sys.stderr.echo = tep_log
else:                                            # chạy bằng python thường
    class GhiCaHai:
        def __init__(self, man_hinh, tep): self.man_hinh, self.tep = man_hinh, tep
        def write(self, s): self.man_hinh.write(s); self.tep.write(s); return len(s)
        def flush(self): self.man_hinh.flush(); self.tep.flush()
        def __getattr__(self, ten): return getattr(self.man_hinh, ten)
    sys.stdout = GhiCaHai(sys.__stdout__, tep_log)

logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logging.captureWarnings(True)
log = logging.getLogger(TEN_NOTEBOOK)
log.info(f"Bắt đầu chạy {TEN_NOTEBOOK} – người chạy: {NGUOI_CHAY} – file log: {file_log}")

2026-11-14 00:00:10 | INFO | Bắt đầu chạy 00_E0.1_cach_ghi_va_thoi_gian_Han – người chạy: Han – file log: /Users/caongochan/Documents/Nhom11_DuBaoDienNang_GiuaKy/logs/eda_log/2026-11-14_0000_00_E0.1_cach_ghi_va_thoi_gian_Han.log


# E0.1 – Dữ liệu được ghi thế nào?

| | |
|---|---|
| **Tác giả** | Cao Ngọc Hân – Nhóm 11 |
| **Ngày tạo** | 13/11/2026 |
| **Thí nghiệm** | E0.1 – Dữ liệu được ghi thế nào? |
| **Mục đích** | Xem dữ liệu thô mức phút được lưu ra sao: kiểu cột, cách ghi giá trị thiếu, trục thời gian, độ phủ tháng đầu và tháng cuối. |
| **Đọc vào** | `data/raw/individual+household+electric+power+consumption.zip` (đọc thẳng từ zip) |
| **Ghi ra** | `data/eda_data/eda_00_cach_ghi_thieu.csv`, `data/eda_data/eda_E0.1_kho_khan.csv` |
| **Chạy sau** | không cần (chạy đầu tiên) |

Notebook EDA chỉ đọc và mô tả dữ liệu, không sửa, không xoá, không điền dữ liệu. Bản gộp đọc từ trên xuống: `00_EDA_tong_hop_Han.ipynb`.

In [2]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ZIP = ROOT / "data" / "raw" / "individual+household+electric+power+consumption.zip"
TXT = "household_power_consumption.txt"
DIR_DATA, DIR_FIG = ROOT / "data" / "eda_data", ROOT / "figures" / "eda_figures"
DIR_FIG.mkdir(exist_ok=True)

MEAN_COLS = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity"]
SUB_COLS = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
COLS = MEAN_COLS + SUB_COLS
TARGET = "Global_active_power"
DON_VI = {"Global_active_power": "kW", "Global_reactive_power": "kVAR", "Voltage": "V",
          "Global_intensity": "A", "Sub_metering_1": "Wh/phút", "Sub_metering_2": "Wh/phút",
          "Sub_metering_3": "Wh/phút"}
SEED = 42

def fmt(n):
    return f"{int(n):,}".replace(",", ".")

def runs(mask):
    """Các đoạn True liên tiếp: trả về (vị trí bắt đầu, độ dài)."""
    m = np.concatenate(([0], np.asarray(mask).astype(np.int8), [0]))
    d = np.diff(m)
    return np.where(d == 1)[0], np.where(d == -1)[0] - np.where(d == 1)[0]

kho_khan = []   # bảng "khó khăn dữ liệu → cách xử lý", điền dần qua từng mục
def ghi_kho_khan(kho_khan_, bang_chung, cach_xu_ly, o_dau):
    kho_khan.append({"Khó khăn": kho_khan_, "Bằng chứng (số thật)": bang_chung,
                     "Cách xử lý": cach_xu_ly, "Xử lý ở đâu": o_dau})

log.info(f"ROOT: {ROOT}")
log.info(f"File dữ liệu: {ZIP.name} ({ZIP.stat().st_size / 1024**2:.2f} MB)")

2026-11-14 00:00:10 | INFO | ROOT: /Users/caongochan/Documents/Nhom11_DuBaoDienNang_GiuaKy


2026-11-14 00:00:10 | INFO | File dữ liệu: individual+household+electric+power+consumption.zip (19.68 MB)


In [3]:
MA_TN = "E0.1"
moc_kk = len(kho_khan)   # các khó khăn ghi từ đây trở đi thuộc E0.1

## 1. Dữ liệu được ghi như thế nào?
Đọc **mọi cột ở dạng chuỗi** (chưa đổi kiểu) để thấy đúng cách giá trị được lưu trong file gốc. Sau đó thử đọc theo mặc định của pandas để xem chuyện gì xảy ra nếu không xem dữ liệu trước.

In [4]:
with zipfile.ZipFile(ZIP) as z:
    raw_str = pd.read_csv(z.open(TXT), sep=";", dtype=str, keep_default_na=False)
    raw_macdinh = pd.read_csv(z.open(TXT), sep=";", low_memory=False)

n_dong, n_cot = raw_str.shape
log.info(f"Số dòng: {fmt(n_dong)} | Số cột: {n_cot}")
print("5 dòng đầu:")
display(raw_str.head())
print("Kiểu cột nếu đọc theo mặc định của pandas (không khai báo giá trị thiếu):")
print(raw_macdinh.dtypes.astype(str).to_string())
cot_bi_chuoi = [c for c in COLS if raw_macdinh[c].dtype == object or str(raw_macdinh[c].dtype) == "str"]
log.info(f"{len(cot_bi_chuoi)}/7 cột đo bị đọc thành chuỗi thay vì số: {cot_bi_chuoi}")
del raw_macdinh

2026-11-14 00:00:12 | INFO | Số dòng: 2.075.259 | Số cột: 9


5 dòng đầu:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.000
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.000
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.000
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.000
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.000


Kiểu cột nếu đọc theo mặc định của pandas (không khai báo giá trị thiếu):
Date                         str
Time                         str
Global_active_power          str
Global_reactive_power        str
Voltage                      str
Global_intensity             str
Sub_metering_1               str
Sub_metering_2               str
Sub_metering_3           float64
2026-11-14 00:00:12 | INFO | 6/7 cột đo bị đọc thành chuỗi thay vì số: ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2']


**Nhận xét:** 

 Dữ liệu có 2.075.259 dòng và 9 cột, mỗi dòng là số đo của 1 phút. Khi đọc mặc định, 6/7 cột đo bị đọc thành chuỗi vì trong cột có dấu ? (giá trị thiếu), nên phải khai báo ? là giá trị thiếu thì mới đọc thành số được.

## 2. Giá trị thiếu được ghi bằng gì?
Đếm riêng 2 kiểu: ký tự `?` và ô để trống.

In [5]:
dem_hoi = (raw_str[COLS] == "?").sum()
dem_trong = (raw_str[COLS] == "").sum()
bang_thieu = pd.DataFrame({"so_dau_hoi": dem_hoi, "so_o_trong": dem_trong})
bang_thieu["tong_thieu"] = bang_thieu.sum(axis=1)
bang_thieu["ti_le_%"] = (bang_thieu["tong_thieu"] / n_dong * 100).round(3)
bang_thieu.to_csv(DIR_DATA / "eda_00_cach_ghi_thieu.csv", encoding="utf-8-sig")
display(bang_thieu)

thieu = raw_str[COLS].isin(["?", ""])
so_cot_thieu_moi_dong = thieu.sum(axis=1).value_counts().sort_index()
print("Số cột bị thiếu trên mỗi dòng → số dòng:")
print(so_cot_thieu_moi_dong.to_string())
thieu_dong_thoi = set(so_cot_thieu_moi_dong.index) <= {0, len(COLS)}
log.info(f"Thiếu đồng thời cả 7 cột trên cùng một dòng: {thieu_dong_thoi}")

ghi_kho_khan("Giá trị thiếu ghi theo 2 kiểu khác nhau",
             f"6 cột ghi thiếu bằng '?' ({fmt(dem_hoi.max())} dòng mỗi cột); Sub_metering_3 để trống ({fmt(dem_trong['Sub_metering_3'])} dòng). "
             f"Đọc mặc định thì {len(cot_bi_chuoi)}/7 cột đo thành chuỗi",
             "Khai báo na_values='?' khi đọc để mọi cột đo thành số, ô trống tự thành NaN",
             "01_cleaning, mục 1")

,so_dau_hoi,so_o_trong,tong_thieu,ti_le_%
Global_active_power,25979,0,25979,1.252
Global_reactive_power,25979,0,25979,1.252
Voltage,25979,0,25979,1.252
Global_intensity,25979,0,25979,1.252
Sub_metering_1,25979,0,25979,1.252
Sub_metering_2,25979,0,25979,1.252
Sub_metering_3,0,25979,25979,1.252


Số cột bị thiếu trên mỗi dòng → số dòng:
0    2049280
7      25979
2026-11-14 00:00:13 | INFO | Thiếu đồng thời cả 7 cột trên cùng một dòng: True


**Nhận xét:** 

Mỗi cột đo thiếu 25.979 dòng (1,252%). Có hai cách ghi thiếu khác nhau: 6 cột dùng dấu ?, riêng Sub_metering_3 để ô trống. Các dòng thiếu thì thiếu cùng lúc cả 7 cột, tức là lúc đó máy đo ngừng ghi chứ không phải hỏng riêng một cột.

## 3. Trục thời gian
Date và Time là 2 cột chuỗi riêng. Kiểm tra định dạng, khoảng thời gian, bước thời gian giữa 2 dòng liên tiếp, dòng trùng và độ phủ của tháng đầu, tháng cuối.

In [6]:
print("Ví dụ Date:", raw_str["Date"].iloc[[0, 1000, -1]].tolist(), "| Time:", raw_str["Time"].iloc[[0, -1]].tolist())
ts = pd.to_datetime(raw_str["Date"] + " " + raw_str["Time"], format="%d/%m/%Y %H:%M:%S")
buoc = ts.diff().dropna().value_counts()
log.info(f"Từ {ts.iloc[0]:%d/%m/%Y %H:%M} đến {ts.iloc[-1]:%d/%m/%Y %H:%M}")
log.info(f"Bước thời gian giữa 2 dòng liên tiếp: {dict((str(k), int(v)) for k, v in buoc.items())}")
log.info(f"Mốc thời gian trùng: {int(ts.duplicated().sum())} | Dòng trùng hoàn toàn: {int(raw_str.duplicated().sum())}")

so_phut_thang = ts.dt.to_period("M").value_counts().sort_index()
thang_dau, thang_cuoi = so_phut_thang.index[0], so_phut_thang.index[-1]
day_du = lambda p: p.days_in_month * 24 * 60
log.info(f"Tháng đầu {thang_dau}: {fmt(so_phut_thang.iloc[0])}/{fmt(day_du(thang_dau))} phút "
         f"({so_phut_thang.iloc[0] / day_du(thang_dau):.0%}); tháng cuối {thang_cuoi}: "
         f"{fmt(so_phut_thang.iloc[-1])}/{fmt(day_du(thang_cuoi))} phút ({so_phut_thang.iloc[-1] / day_du(thang_cuoi):.0%})")

ghi_kho_khan("Thời gian lưu ở 2 cột chuỗi, định dạng ngày kiểu Pháp",
             f"Date dạng dd/mm/yyyy, Time dạng hh:mm:ss; {fmt(n_dong)} dòng, bước 1 phút, 0 mốc trùng",
             "Gộp Date và Time thành datetime với format '%d/%m/%Y %H:%M:%S' (không để pandas tự đoán, tránh lẫn ngày/tháng)",
             "01_cleaning, mục 1")
ghi_kho_khan("Tháng đầu và tháng cuối không đủ",
             f"{thang_dau} chỉ có {so_phut_thang.iloc[0] / day_du(thang_dau):.0%} số phút, {thang_cuoi} có {so_phut_thang.iloc[-1] / day_du(thang_cuoi):.0%}",
             "Không so sánh trực tiếp 2 tháng này với tháng khác; khi tính trung bình theo năm dùng năm vận hành tháng 12 đến tháng 11",
             "02_E0.6 (xu hướng dài hạn)")

Ví dụ Date: ['16/12/2006', '17/12/2006', '26/11/2010'] | Time: ['17:24:00', '21:02:00']


2026-11-14 00:00:16 | INFO | Từ 16/12/2006 17:24 đến 26/11/2010 21:02


2026-11-14 00:00:16 | INFO | Bước thời gian giữa 2 dòng liên tiếp: {'0 days 00:01:00': 2075258}


2026-11-14 00:00:17 | INFO | Mốc thời gian trùng: 0 | Dòng trùng hoàn toàn: 0


2026-11-14 00:00:17 | INFO | Tháng đầu 2006-12: 21.996/44.640 phút (49%); tháng cuối 2010-11: 37.263/43.200 phút (86%)


**Nhận xét:** 

Dữ liệu kéo dài từ 16/12/2006 17:24 đến 26/11/2010 21:02. Các dòng cách nhau đúng 1 phút, không có mốc thời gian trùng, nên trục thời gian sạch. Tuy nhiên tháng đầu chỉ có 49% số phút và tháng cuối có 86%, nên khi so sánh theo tháng cần lưu ý hai tháng này không đủ. 

## Khó khăn dữ liệu rút ra từ E0.1

In [7]:
bang_kk = pd.DataFrame(kho_khan[moc_kk:])
bang_kk.to_csv(DIR_DATA / f"eda_{MA_TN}_kho_khan.csv", encoding="utf-8-sig", index=False)
with pd.option_context("display.max_colwidth", None):
    display(bang_kk)
log.info(f"{MA_TN}: {len(bang_kk)} khó khăn dữ liệu, lưu ở data/eda_data/eda_{MA_TN}_kho_khan.csv")

,Khó khăn,Bằng chứng (số thật),Cách xử lý,Xử lý ở đâu
0,Giá trị thiếu ghi theo 2 kiểu khác nhau,6 cột ghi thiếu bằng '?' (25.979 dòng mỗi cột); Sub_metering_3 để trống (25.979 dòng). Đọc mặc định thì 6/7 cột đo thành chuỗi,"Khai báo na_values='?' khi đọc để mọi cột đo thành số, ô trống tự thành NaN","01_cleaning, mục 1"
1,"Thời gian lưu ở 2 cột chuỗi, định dạng ngày kiểu Pháp","Date dạng dd/mm/yyyy, Time dạng hh:mm:ss; 2.075.259 dòng, bước 1 phút, 0 mốc trùng","Gộp Date và Time thành datetime với format '%d/%m/%Y %H:%M:%S' (không để pandas tự đoán, tránh lẫn ngày/tháng)","01_cleaning, mục 1"
2,Tháng đầu và tháng cuối không đủ,"2006-12 chỉ có 49% số phút, 2010-11 có 86%",Không so sánh trực tiếp 2 tháng này với tháng khác; khi tính trung bình theo năm dùng năm vận hành tháng 12 đến tháng 11,02_E0.6 (xu hướng dài hạn)


2026-11-14 00:00:17 | INFO | E0.1: 3 khó khăn dữ liệu, lưu ở data/eda_E0.1_kho_khan.csv


## Log lên W&B (minh chứng)

In [8]:
# W&B (project cuoi-ky-du-bao-dien-nang). Mặc định TẮT để chạy lại không tạo run trùng.
# Muốn ghi lên W&B: chạy notebook bằng lệnh  BAT_WANDB=1 jupyter nbconvert --to notebook --execute --inplace <notebook>
import os
USE_WANDB = os.environ.get("BAT_WANDB") == "1"
if USE_WANDB:
    import wandb, pandas as pd
    BANG = ['eda_00_cach_ghi_thieu.csv', 'eda_E0.1_kho_khan.csv']
    HINH = []
    run = wandb.init(entity="sgu-mayhoc-nhom11", project="cuoi-ky-du-bao-dien-nang",
                     name="ck-E0.1-Han-cach-ghi-va-thoi-gian", tags=["E0.1", "Han", "EDA"],
                     config={"notebook": TEN_NOTEBOOK, "file_log": str(file_log)})
    run.summary.update({"so_dong": n_dong, "so_cot": n_cot, "so_cot_doc_thanh_chuoi": len(cot_bi_chuoi), "so_dong_thieu_ca_7_cot": int(thieu.all(axis=1).sum())})
    run.log({**{"bang/" + b.removesuffix(".csv"): wandb.Table(dataframe=pd.read_csv(DIR_DATA / b)) for b in BANG},
             **{"hinh/" + h.removesuffix(".png"): wandb.Image(str(DIR_FIG / h)) for h in HINH},
             "ket_luan": wandb.Html("<br>".join(str(x) for x in kho_khan[moc_kk:]))})
    log.info(f"Đã ghi W&B: {run.url}")
    run.finish()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/caongochan/.netrc.


wandb: Currently logged in as: caongochan142 (sgu-mayhoc-nhom11) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: setting up run 1hjdxu7t


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /Users/caongochan/Documents/Nhom11_DuBaoDienNang_GiuaKy/notebooks/eda/wandb/run-20261114_000020-1hjdxu7t
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run ck-E0.1-Han-cach-ghi-va-thoi-gian


... (7 outputs left out)

wandb: uploading media/table/bang/eda_00_cach_ghi_thieu_0_e254e8875fbad92b3d46.table.json


wandb: 
wandb: Run summary:
wandb:                 so_cot 9
wandb: so_cot_doc_thanh_chuoi 6
wandb:                so_dong 2075259
wandb: so_dong_thieu_ca_7_cot 25979
wandb: 


wandb: 🚀 View run ck-E0.1-Han-cach-ghi-va-thoi-gian at: https://forge.coreweave.com/wandb/sgu-mayhoc-nhom11/cuoi-ky-du-bao-dien-nang/runs/1hjdxu7t
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/sgu-mayhoc-nhom11/cuoi-ky-du-bao-dien-nang
wandb: Synced 5 W&B file(s), 3 media file(s), 4 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261114_000020-1hjdxu7t/logs


## Nhật ký

- 13/11/2026 – Viết lần đầu chung trong notebook `00_eda_du_lieu_tho_Han.ipynb` (nay ở `phien_ban_cu/`) rồi tách ra thành notebook riêng `00_E0.1_cach_ghi_va_thoi_gian_Han.ipynb`, có ô ghi log. Commit `15b72c2`.
- 13/11/2026 – Chạy 3 lần (19:28, 20:10, 20:14). Lần đầu chạy xong nhưng đóng tab chưa lưu nên notebook mất output; lần cuối chạy bằng `jupyter nbconvert` để lưu output. Nội dung 3 file log trùng nhau (chỉ khác giờ chạy), tức là kết quả tái lập được. Commit `0dcd35f`.
- 13/11/2026 – Chuyển các file log EDA vào `logs/eda_log/` và sửa ô ghi log để các lần chạy sau ghi thẳng vào thư mục này. Commit `44b69dd`.